# Tool 8bis: Automatic epoch rejection (channel-first, then epoch)

Automatically cleans the epochs that **7_preprocessing** flagged, using a **channel-first then epoch**
rule: a globally-bad channel is dropped *before* the epoch vote, so it no longer
condemns every epoch it appears in.

- **Inputs**: the study folder, where the tool finds tool 7's preprocessed epochs (`{file_id}_all-epo.fif`, e.g.
  `derivatives/preproc_epo`) and its reports (`{file_id}_epoch_channel_rejection.tsv`, e.g. `reports_preproc`), both
  read-only.
- **Outputs**: the decision tables `*_epoch_decision.tsv` and `*_channel_decision.tsv` in `rejection_auto/` (beside
  the preprocessed-epochs folder); the decision record, the HTML report and the global summary in
  `reports_rejection_auto/` (beside the reports folder).
- **Sections**: 1. Study folder (and Scan), 2. Parameters (methods, thresholds and reference stages), 3. Run.

**The decision covers every epoch.** Which stages to analyse is chosen later, by the analysis tools that
follow (tool 9 and the next ones), so one decision serves every analysis. The reference stages only decide which epochs *judge* the channels (and the
participant exclusion), so that wake movement / EMG artefacts do not drop a channel that is clean in sleep.

<details style="border:1px solid #bbb; border-radius:4px; background:#f7f7f7; padding:8px 12px; margin:10px 0;"><summary style="cursor:pointer; font-weight:bold;">Details <span style="font-size:85%; font-weight:normal; color:#666;">(click to open)</span></summary>

**Per participant:** ① a **channel is dropped** when flagged in more than *Channel reject %* of the
epochs of the **reference stages** → ② **every epoch** (whatever its stage) is rejected when flagged in
more than *Epoch reject %* of the remaining **good** channels → ③ the decision is written as two tables,
`*_epoch_decision.tsv` (every epoch: in scope = always True, rejected, why) and `*_channel_decision.tsv` (every channel: dropped, why), to `rejection_auto/`
**beside the preprocessed-epochs folder**, and the reports (decision record, HTML, global summary) to
`reports_rejection_auto/` **beside the reports folder**: following the toolkit's `derivatives/` (data) vs
`reports_*` (reports) split. No epoch is copied: the analysis tools read tool 7's `*_all-epo.fif` and these tables,
so a decision can be changed (re-run with other thresholds) without rewriting any signal.

**Inputs (read-only):** two folders are selected **explicitly**, the **preprocessed-epochs folder** (holding the
tool-7 `{file_id}_all-epo.fif`, e.g. `derivatives/preproc_epo`) and the **reports folder** (holding
`{file_id}_epoch_channel_rejection.tsv`, e.g. `reports_preproc`). Picking them explicitly (rather than
scanning one root recursively) keeps several tool-7 runs apart when their outputs have been renamed/versioned
(`preproc_epo_v1`, `reports_preproc_v2`, …). Tool-7 outputs are never modified. No channel interpolation:
bad channels are dropped (interpolation is a later step).

</details>

In [ ]:
import os, sys, io, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from ipyfilechooser import FileChooser
import mne
mne.set_log_level('ERROR')

# Reuse the shared tool-8 library (find_participants, save_report_html, custom-stage styling). Search
# several locations so it is found whether Voila is launched from the repo root (lib in tools/) or from
# tools_curry/ (the Curry twin lives there; the shared lib stays in ../tools).
_here = os.getcwd()
_lib_candidates = [_here, os.path.join(_here, 'tools'),
                   os.path.join(os.path.dirname(_here), 'tools'), os.path.join(_here, '..', 'tools')]
for _cand in _lib_candidates:
    if os.path.isfile(os.path.join(_cand, 'qc_rejected_epochs_lib.py')):
        _cand = os.path.abspath(_cand)
        if _cand not in sys.path:
            sys.path.insert(0, _cand)
        break
import qc_rejected_epochs_lib as L
import participant_selection_lib as PSL     # exclusion registry + participant selector (same folder)
import ui_lib as UI                         # the look shared by the Voila pages (same folder)
import study_files_lib as SF                # where the files of the study are (same folder)
# The decision rule, its tables and figures, shared with tool 0: tools/autoreject_lib.py (same folder as
# qc_rejected_epochs_lib). Edit the library, never a copy.
from autoreject_lib import (
    read_event_counts, build_pair_matrix, auto_reject_decision, decision_summary_row, read_tool7_bad_channels,
    build_epoch_decision, build_channel_decision, wrap_scroll, decision_overview_html, plot_decision_heatmap,
    ordered_stages, default_reference_stages, event_exclusion_reason, registry_reason, AASM_STAGES,
    settings_differences, event_thresholds_text,
    DEFAULT_CHANNEL_REJECT_PCT, DEFAULT_EPOCH_REJECT_PCT,
)

# The decision tables live beside the data (derivatives/rejection_auto/<subtree>/), the reports in
# reports_rejection_auto/: tool 9 reads tool 7's *_all-epo.fif + these tables, so a decision can be changed
# without rewriting any epochs.
DECISION_DIRNAME = 'rejection_auto'
REPORTS_DIRNAME = 'reports_rejection_auto'

S = {}   # shared state across sections


def registry_folder():
    """Where the exclusion registry lives: the study folder when picked, else the folder holding the
    derivatives/ the preprocessed-epochs folder sits in."""
    if fc_data.selected_path:
        return Path(fc_data.selected_path)
    return PSL.find_data_folder(S.get('raw_root'))


<hr style="height:4px; background-color:black; border:none;">

## 1. Study folder

Pick the **study folder**: the tool finds tool 7's run in it and lists it under **Inputs found**: the
**preprocessed-epochs folder** (`{file_id}_all-epo.fif`, e.g. `derivatives/preproc_epo`) and the **reports folder**
paired with it by name (`{file_id}_epoch_channel_rejection.tsv`, e.g. `reports_preproc`). With several runs (tool 7's
*New version*, `preproc_epo_v2`), a menu lists them with their number of participants, the most recent pre-chosen.
*Choose another…* on a line picks another folder by hand. Then click **Scan**. The decision tables are written to
`rejection_auto/` (beside the preprocessed-epochs folder), the reports to `reports_rejection_auto/` (beside the reports
folder). These names carry the suffix of the run (`rejection_auto_v2`, `reports_rejection_auto_v2`), and an automatic
exclusion holds for this run only.

In [ ]:
fc_data = FileChooser(os.getcwd())
fc_data.show_only_dirs = True
fc_data.title = UI.picker_title('Study folder', '(holds <code>derivatives/</code> + <code>reports_preproc/</code>)')

# The input folders, found in the study folder (the 'Inputs found' list): each picker shows under its line, with
# "Choose another…", for a folder chosen by hand. The scan reads their selection.
fc_raw = FileChooser(os.getcwd())
fc_raw.show_only_dirs = True
fc_raw.title = UI.picker_title('Preprocessed-epochs folder', '(holds the tool-7 <code>*_all-epo.fif</code>, e.g. <code>derivatives/preproc_epo</code>)')

fc_reports = FileChooser(os.getcwd())
fc_reports.show_only_dirs = True
fc_reports.title = UI.picker_title('Reports folder', '(holds <code>*_epoch_channel_rejection.tsv</code>, e.g. <code>reports_preproc</code>)')

btn_scan = UI.button('Scan', 'read')
lbl_scan = widgets.HTML('<i>Pick the study folder, then Scan.</i>')


def _pair_reports(chooser=None):
    """The reports folder paired by name with the epochs folder (found, chosen in the menu or by hand)."""
    if not fc_data.selected_path:
        return
    data = Path(fc_data.selected_path)
    reports = SF.paired_reports_folder(fc_raw.selected_path) if fc_raw.selected_path else None
    locate_reports(data, reports, 'paired with the epochs folder', data)


def _locate_run(folder, how):
    """The run of tool 7 to read (its reports folder follows, through _pair_reports)."""
    data = Path(fc_data.selected_path)
    locate_raw(data, folder, how, data / 'derivatives' if (data / 'derivatives').is_dir() else data)


raw_row, locate_raw = UI.input_row(fc_raw, 'Preprocessed epochs', 'required', 'run tool 7 first',
                                   check=SF.check_epoch_folder, on_change=_pair_reports)
reports_row, locate_reports = UI.input_row(fc_reports, 'Tool 7 reports', 'required', 'run tool 7 first, or choose them',
                                           check=SF.check_reports_folder)
# several runs of tool 7 in the study: a menu, with the number of participants of each
run_menu, fill_run_menu = UI.choice_menu('Tool 7 run:', lambda value: _locate_run(Path(value), 'chosen in the menu'))
inputs_list, show_inputs = UI.inputs_list([run_menu, raw_row, reports_row])


def _on_data(chooser):
    """Find tool 7's run in the study folder (the 'Inputs found' list)."""
    try:
        data = fc_data.selected_path
        if not data:
            show_inputs(False)
            fill_run_menu([])
            locate_raw(None)
            locate_reports(None)
            return
        data = Path(data)
        show_inputs(True)
        names, names_warn = SF.load_names(data)
        folders = SF.epoch_folders(data, names)
        fill_run_menu(SF.epoch_folder_options(data, folders))
        _locate_run(folders[0][0] if folders else None,
                    'the only run found' if len(folders) == 1 else 'the most recent run')
        lbl_scan.value = UI.msg('warn', names_warn) if names_warn else '<i>Study folder set: check the folders, then Scan.</i>'
    except Exception as e:
        lbl_scan.value = f'<span style="color:{UI.ERROR_COLOR}">Study folder error: {e}</span>'


def _scan(b):
    try:
        raw, reports = fc_raw.selected_path, fc_reports.selected_path
        if not raw or not reports:
            lbl_scan.value = UI.msg('error', 'Preprocessed epochs or reports missing: run tool 7 first, or choose '
                                             'them with "Choose another…".'); return
        raw_root, reports_root = Path(raw), Path(reports)
        parts = L.find_participants(raw_root)
        # Locate each participant's flags TSV + the optional event sidecars in the chosen reports
        # folder (once; reused by the run). Also collect the methods present and the event types.
        tsv_by_fid = {}
        flags_by_fid = {}      # {fid: _event_epoch_flags.tsv path or None}
        counts_by_fid = {}     # {fid: _event_counts.tsv path or None}
        stages = set(); methods_present = set(); event_types = set()
        type_counts = {}       # {event_type: [n_events per file that has it]} -> mean/median hints
        for p in parts:
            fid = p['file_id']
            tsv = next(reports_root.rglob(f'{fid}_epoch_channel_rejection.tsv'), None)
            tsv_by_fid[fid] = tsv
            if tsv is not None:
                try:
                    cols = pd.read_csv(tsv, sep='\t', nrows=0).columns
                    methods_present.update(m for m in L.METHOD_ORDER if ('flag_' + m) in cols)
                    stages.update(pd.read_csv(tsv, sep='\t', usecols=['stage'])['stage'].astype(str).unique())
                except Exception:
                    pass
            ef = next(reports_root.rglob(f'{fid}_event_epoch_flags.tsv'), None)
            flags_by_fid[fid] = ef
            if ef is not None:
                try:
                    event_types.update(c[len('evt_'):] for c in pd.read_csv(ef, sep='\t', nrows=0).columns
                                       if c.startswith('evt_'))
                except Exception:
                    pass
            ec = next(reports_root.rglob(f'{fid}_event_counts.tsv'), None)
            counts_by_fid[fid] = ec
            if ec is not None:
                try:
                    ecdf = pd.read_csv(ec, sep='\t')
                    event_types.update(ecdf['event_type'].astype(str).unique())
                    for _, r in ecdf.iterrows():
                        type_counts.setdefault(str(r['event_type']), []).append(int(r['n_events']))
                except Exception:
                    pass
        ordered = ordered_stages(stages)
        S['raw_root'] = raw_root; S['reports_root'] = reports_root
        S['parts'] = parts; S['tsv_by_fid'] = tsv_by_fid
        S['flags_by_fid'] = flags_by_fid; S['counts_by_fid'] = counts_by_fid
        # ---- Stage checkboxes (default ticked: N2/N3/R, else all) ----
        prefer = default_reference_stages(ordered)
        S['stage_checks'] = {}
        boxes = []
        for st in ordered:
            cb = widgets.Checkbox(value=(st in prefer), description=st,
                                  indent=False, layout=widgets.Layout(width='75px'))
            S['stage_checks'][st] = cb
            boxes.append(cb)
        stage_box.children = boxes
        # ---- Flagging-method checkboxes (all ON by default = today's reject_any) ----
        S['method_checks'] = {}
        mboxes = []
        for m in [m for m in L.METHOD_ORDER if m in methods_present]:
            cb = widgets.Checkbox(value=True, description=L.METHOD_LABEL.get(m, m),
                                  indent=False, layout=widgets.Layout(width='120px'))
            S['method_checks'][m] = cb
            mboxes.append(cb)
        method_box.children = mboxes
        # ---- Event-type rows (checkbox + per-type exclusion threshold), nested under the 'event' method ----
        S['event_type_rows'] = {}
        erows = []
        for t in sorted(event_types):
            cbt = widgets.Checkbox(value=True, description=t, indent=False,
                                   layout=widgets.Layout(width='190px'))
            thr = widgets.IntText(value=0, layout=widgets.Layout(width='80px'))
            # Per-file event-count hint across the files that HAVE this type (guides the threshold).
            vals = type_counts.get(t, [])
            hint = (f'&nbsp;<small style="color:{UI.MUTED_COLOR};">mean {np.mean(vals):.1f} (median '
                    f'{np.median(vals):g}) events/file, n={len(vals)}</small>') if vals else ''
            erows.append(widgets.HBox([cbt, widgets.HTML('exclude participant if &gt;'), thr,
                                       widgets.HTML('events (0 = off)'), widgets.HTML(hint)]))
            S['event_type_rows'][t] = (cbt, thr)
        event_types_box.children = erows
        # Reveal the event-type rows only when the 'event' method is ticked (initial display from its value).
        ev_master = S['method_checks'].get('event')
        if ev_master is not None:
            def _toggle_events(change=None):
                event_section.layout.display = '' if ev_master.value else 'none'
            ev_master.observe(_toggle_events, names='value')
            _toggle_events()
        else:
            event_section.layout.display = 'none'   # no event flagging in any scanned file
        n_tsv = sum(1 for t in tsv_by_fid.values() if t is not None)
        colour = UI.OK_COLOR if (parts and n_tsv == len(parts)) else (UI.ERROR_COLOR if not parts else UI.WARN_COLOR)
        msg = f'{len(parts)} participant(s) in the raw folder; {n_tsv} with a matching TSV in the reports folder.'
        if parts and n_tsv < len(parts):
            msg += ' ⚠️ some participants have no TSV here (version mismatch?).'
        # Participants for the selector: done = decision record (reports) + epoch decision (data) on disk
        run = SF.run_suffix(raw_root)         # the outputs and the registry rows of a run of tool 7 carry its suffix
        reports_out = reports_root.parent / (REPORTS_DIRNAME + run)
        decision_out = raw_root.parent / (DECISION_DIRNAME + run)
        done, older_than_tool7 = [], []
        for p in parts:
            try:
                subtree = Path(p['folder']).relative_to(raw_root)
            except Exception:
                subtree = Path('.')
            if ((reports_out / subtree / f"{p['file_id']}_autoreject_decision.tsv").exists()
                    and (decision_out / subtree / f"{p['file_id']}_epoch_decision.tsv").exists()):
                done.append(p['file_id'])
                if SF.preproc_changed(
                        SF.recorded_preproc_fingerprint(reports_out / subtree / f"{p['file_id']}_autoreject_decision.tsv"),
                        SF.preproc_fingerprint(Path(p['folder']) / f"{p['file_id']}_preprocessing_params.json")):
                    older_than_tool7.append(p['file_id'])
        reg, reg_warn = PSL.load_registry(registry_folder())
        status = PSL.effective_status(reg, run)
        participant_selector.set_participants(
            [p['file_id'] for p in parts], done=done,
            excluded={s['file_id']: s['reason'] for s in status.values() if s['excluded']})
        older_note = ('<br>' + UI.msg('warn', f'{len(older_than_tool7)} participant(s) decided before tool 7 was run '
                                              'again with other settings: Run will ask about them.')
                      if older_than_tool7 else '')
        if reg_warn:
            msg += f' ⚠️ {reg_warn}'
        stale = (PSL.stale_tables_html(PSL.STALE_RERUN)
                 if not reg_warn and PSL.tables_out_of_date(status, reports_out / 'global_autoreject_summary.tsv') else '')
        lbl_scan.value = (f'<span style="color:{colour}">{msg}</span>{older_note}'
                          f'<br><small>decisions &rarr; {decision_out}'
                          f'<br>reports &rarr; {reports_out}</small>' + stale)
    except Exception as e:
        lbl_scan.value = f'<span style="color:{UI.ERROR_COLOR}">Scan error: {e}</span>'


fc_data.register_callback(_on_data)
btn_scan.on_click(_scan)
display(widgets.VBox([fc_data, inputs_list, btn_scan, lbl_scan]))


<hr style="height:4px; background-color:black; border:none;">

## 2. Parameters

A **channel** is dropped when flagged in more than *Channel reject %* of the epochs of the **reference
stages**. Then **every epoch**, whatever its stage, is rejected when flagged in more than *Epoch reject %*
of the remaining **good** channels. Both default to 20% and are editable. The reference stages (default
N2/N3/R) judge the channels and the participant exclusion; the decision itself covers every epoch
(`in_scope = True` everywhere), the stages to analyse being chosen later by the analysis tools that follow.

In [ ]:
ft_ch_pct = widgets.FloatText(value=DEFAULT_CHANNEL_REJECT_PCT, description='Channel reject > (%)',
                              style=UI.LABEL_STYLE, layout=widgets.Layout(width=UI.SHORT_WIDTH))
ft_ep_pct = widgets.FloatText(value=DEFAULT_EPOCH_REJECT_PCT, description='Epoch reject > (%)',
                              style=UI.LABEL_STYLE, layout=widgets.Layout(width=UI.SHORT_WIDTH))
# All three boxes below are populated when you Scan (Section 1): one checkbox per detected stage,
# one per flagging method present, and one row (checkbox + exclusion threshold) per event type.
stage_box = widgets.Box([], layout=widgets.Layout(display='flex', flex_flow='row wrap',
                                                  width='100%'))
method_box = widgets.Box([], layout=widgets.Layout(display='flex', flex_flow='row wrap', width='100%'))
# Nested under the 'event' method checkbox: revealed only when 'event' is ticked (wired at Scan time).
event_types_box = widgets.VBox([], layout=widgets.Layout(margin='2px 0 0 25px'))
event_section = widgets.VBox([
    widgets.HTML(UI.text_block('<b>Event types</b>: tick the types allowed to reject an epoch; set a '
                 'threshold to <b>exclude the whole participant</b> when the file has more events of '
                 'that type than the threshold (0 = no exclusion). Needs the tool-7 '
                 '<code>*_event_epoch_flags.tsv</code> / <code>*_event_counts.tsv</code> sidecars; '
                 'files preprocessed before this feature fall back to the whole <code>flag_event</code> '
                 '(all event types, no sub-selection).')),
    event_types_box,
])

display(widgets.VBox([
    ft_ch_pct, ft_ep_pct,
    widgets.HTML(UI.help_html('The 20 % defaults suit a typical clinical PSG. Adapt them to the quality of your '
                              'dataset: on a noisy one they reject much of the night, on a very clean one almost '
                              'nothing. Check the result in the per-participant reports.')),
    widgets.HTML(UI.text_block('<b>Flagging methods</b> (populated at Scan): tick the tool-7 methods that '
                 'drive the rejection: all on reproduces the previous behaviour, but after inspecting '
                 'the tool-7 flagging reports you can drop one that looks unhelpful. Ticking '
                 '<b>event</b> reveals the event types below.')),
    method_box, event_section,
    widgets.HTML(UI.text_block('<b>Reference stages</b>: the stages that judge the channels (channel badness) and '
                 'the participant exclusion. The epoch rule is then applied to <b>every</b> epoch, '
                 'whatever its stage (the stages to analyse are chosen by the analysis tools that follow):')),
    stage_box,
]))


<hr style="height:4px; background-color:black; border:none;">

## 3. Run

Loops over the participants chosen below (all, or a subset). `{file_id}_epoch_decision.tsv` +
`{file_id}_channel_decision.tsv` go to `rejection_auto/<subtree>/` (beside the raw folder). `{file_id}_autoreject_decision.tsv` +
`{file_id}_autoreject_report.html` + the rebuilt `global_autoreject_summary.tsv` go to
`reports_rejection_auto/` (beside the reports folder). A **summary table** is shown at the end. A
participant is
**skipped** if it already has both its epoch decision and its decision record **made with the same settings**
(thresholds, reference stages, methods, event types and their thresholds): a participant decided with other
settings is decided again, as the log says. Uncheck *Skip* to reprocess everyone.
Participants where **every channel** (or every reference-stage epoch) would be rejected still get their tables and
a report (shown as **100% rejected** in the summary) and are **excluded automatically** in the exclusion
registry (`8bis_auto`), like a participant over an event-count threshold. A re-run that no longer
excludes them lifts that exclusion.

In [ ]:
participant_selector = PSL.ParticipantSelector()       # all participants / a subset (+ skip)
cb_skip = participant_selector.cb_skip
btn_run = UI.button('Run automatic rejection', 'run')
prog, prog_row = UI.progress_bar('Participants:')
lbl_prog = widgets.HTML(layout=widgets.Layout(margin='0 0 0 8px'))   # current participant (i/N)
out_run = widgets.Output()


def _rebuild_global_summary(auto_root):
    files = sorted(Path(auto_root).rglob('*_autoreject_decision.tsv'))
    rows = []
    for f in files:
        try:
            rows.append(pd.read_csv(f, sep='\t', dtype={'file_id': str}))
        except Exception:
            pass
    if not rows:
        return pd.DataFrame()
    summary = pd.concat(rows, ignore_index=True)
    # flag (never drop) the participants excluded in the registry (5bis / tools 7, 8, 8bis)
    reg, _warn = PSL.load_registry(registry_folder())
    return PSL.add_exclusion_columns(summary, PSL.effective_status(reg, SF.run_suffix(S['raw_root'])))


def run_autoreject(b, other_settings_answer=None):
    """other_settings_answer: None, or the answer ('reprocess', 'keep') when participants were made with other
    settings than the run's (see the question below)."""
    # A question may stop this run under the bars: they must not still say the previous run is done.
    participant_selector.run_note.value = ''
    prog.value = 0
    prog.bar_style = 'info'
    lbl_prog.value = ''
    with out_run:
        clear_output(wait=True)
        try:
            if 'raw_root' not in S:
                print('Select the raw + reports folders and click Scan first (Section 1).'); return
            raw_root = Path(S['raw_root']); reports_root = Path(S['reports_root'])
            soi = [st for st, cb in S.get('stage_checks', {}).items() if cb.value]
            if not soi:
                print('Tick at least one stage of interest (Section 2).'); return
            ch_pct = float(ft_ch_pct.value) / 100.0
            ep_pct = float(ft_ep_pct.value) / 100.0
            skip = cb_skip.value
            chosen = {PSL.id_key(f) for f in participant_selector.chosen_ids()}
            reg_folder = registry_folder()
            # Section-2 flagging-method + event sub-selection.
            methods_sel = [m for m, cb in S.get('method_checks', {}).items() if cb.value]
            event_master_on = ('event' in methods_sel)
            event_types_sel = [t for t, (cb, thr) in S.get('event_type_rows', {}).items() if cb.value]
            # Per-type exclusion thresholds: only ticked types with a positive threshold are evaluated.
            event_thresholds = {t: int(thr.value) for t, (cb, thr) in S.get('event_type_rows', {}).items()
                                if cb.value and int(thr.value) > 0}
            if not methods_sel:
                print('Tick at least one flagging method (Section 2).'); return
            thresholds_str = event_thresholds_text(event_thresholds) if event_master_on else ''
            flags_by_fid = S.get('flags_by_fid', {}); counts_by_fid = S.get('counts_by_fid', {})
            parts = [p for p in S.get('parts', []) if PSL.id_key(p['file_id']) in chosen]
            tsv_by_fid = S.get('tsv_by_fid', {})
            # DATA (the epoch / channel decision tables) beside the preprocessed-epochs folder; REPORTS (decision
            # record + html + globals) beside the reports folder: the toolkit's derivatives/ vs reports_* split.
            run = SF.run_suffix(raw_root)     # the outputs and the registry rows of a run of tool 7 carry its suffix
            decision_out = raw_root.parent / (DECISION_DIRNAME + run)
            reports_out = reports_root.parent / (REPORTS_DIRNAME + run)

            def output_paths(p):
                """Where this run writes a participant: reports folder, decision folder, record, epoch decision."""
                try:
                    subtree = Path(p['folder']).relative_to(raw_root)   # participant subtree in the raw folder
                except Exception:
                    subtree = Path('.')
                reports_dir = reports_out / subtree                     # holds the decision tsv + html
                decision_dir = decision_out / subtree                   # epoch/channel decisions
                return (reports_dir, decision_dir, reports_dir / f"{p['file_id']}_autoreject_decision.tsv",
                        decision_dir / f"{p['file_id']}_epoch_decision.tsv")

            def flags_tsv(fid):
                return tsv_by_fid.get(fid) or next(reports_root.rglob(f'{fid}_epoch_channel_rejection.tsv'), None)

            def read_flags(p, tsv):
                """A participant's pair matrix from tool 7's flags, and the settings this run would record for it."""
                fid = p['file_id']
                ef_path = flags_by_fid.get(fid) or next(reports_root.rglob(f'{fid}_event_epoch_flags.tsv'), None)
                piv, stage_by_epoch, pm_meta = build_pair_matrix(
                    tsv, methods=methods_sel, event_master=event_master_on,
                    event_types=event_types_sel, event_flags_path=ef_path)
                stages_present = set(stage_by_epoch.dropna().astype(str).unique())
                soi_p = [s for s in soi if s in stages_present]
                settings = dict(
                    channel_reject_pct=round(ch_pct * 100, 3), epoch_reject_pct=round(ep_pct * 100, 3),
                    stages_of_interest='+'.join(soi_p), methods_used='+'.join(pm_meta['used_methods']) or '(none)',
                    event_types_used='+'.join(pm_meta['used_event_types']), event_thresholds=thresholds_str,
                    preproc_fingerprint=SF.preproc_fingerprint(Path(p['folder']) / f'{fid}_preprocessing_params.json'))
                return piv, stage_by_epoch, pm_meta, stages_present, soi_p, settings

            # Skip ticked: participants already decided with other settings are asked about here, before any
            # decision (PSL, Participants made with other settings).
            differences, n_processed, n_new, n_not_recorded = {}, 0, 0, 0
            if skip:
                for p in parts:
                    _reports, _decisions, decision_tsv, epoch_dec_tsv = output_paths(p)
                    if not (epoch_dec_tsv.exists() and decision_tsv.exists()):
                        n_new += 1
                        continue
                    n_processed += 1
                    tsv = flags_tsv(p['file_id'])
                    if tsv is None:           # no flags to compare the settings with: the decision stays
                        n_not_recorded += 1
                        continue
                    try:
                        rows = settings_differences(decision_tsv, read_flags(p, tsv)[-1])
                    except Exception:
                        rows = [('tool 7 flags', 'unreadable', 'this run')]
                    if rows:
                        differences[p['file_id']] = rows
            if differences and other_settings_answer is None:
                participant_selector.ask_other_settings(differences, n_processed, n_new,
                                                        lambda answer: run_autoreject(b, answer))
                return
            redo = {PSL.id_key(f) for f in differences} if other_settings_answer == 'reprocess' else set()
            if participant_selector.warning_html() and not redo:      # a subset with already-processed participants
                display(HTML(participant_selector.warning_html()))
            prog.max = max(1, len(parts)); prog.value = 0; prog.bar_style = 'info'
            failed = []; saved = 0; all_rejected = 0; skipped = 0; excluded = 0
            for i, p in enumerate(parts):
                fid = p['file_id']
                lbl_prog.value = f'<b>{fid}</b> ({i + 1}/{len(parts)})'
                try:
                    reports_dir, decision_dir, decision_tsv, epoch_dec_tsv = output_paths(p)
                    if skip and epoch_dec_tsv.exists() and decision_tsv.exists() and PSL.id_key(fid) not in redo:
                        skipped += 1; prog.value += 1; continue
                    tsv = flags_tsv(fid)
                    if tsv is None:
                        failed.append((fid, 'no _epoch_channel_rejection.tsv in the reports folder'))
                        prog.value += 1; continue
                    ec_path = counts_by_fid.get(fid) or next(reports_root.rglob(f'{fid}_event_counts.tsv'), None)
                    piv, stage_by_epoch, pm_meta, stages_present, soi_p, settings = read_flags(p, tsv)
                    methods_used_str, event_types_used_str = settings['methods_used'], settings['event_types_used']
                    if pm_meta.get('warn'):
                        print(f'  ⚠️ {fid}: {pm_meta["warn"]}')
                    if not soi_p:
                        failed.append((fid, 'none of the selected stages present')); prog.value += 1; continue
                    dec = auto_reject_decision(piv, stage_by_epoch, soi_p, ch_pct, ep_pct)
                    tool7_bads = read_tool7_bad_channels(p['folder'], fid)
                    # Per-event participant exclusion: drop the WHOLE participant when the file has more
                    # events of a ticked type than its threshold (raw event count from _event_counts.tsv).
                    excl_reason = ''
                    if event_master_on and event_thresholds:
                        counts = read_event_counts(ec_path) if ec_path is not None else {}
                        excl_reason = event_exclusion_reason(counts, event_thresholds)
                    # An all-rejected participant (every channel dropped, or every reference-stage epoch rejected)
                    # is a valid extreme outcome: its tables + report are written like any other (100 %
                    # rejected in the summary) and it is excluded in the registry below.
                    has_output = ((not excl_reason) and bool(dec['good_channels'])
                                  and dec['n_rejected_ref'] < dec['n_ref'])
                    reports_dir.mkdir(parents=True, exist_ok=True)
                    # --- DATA first: the epoch + channel decision tables (what tool 9 joins to the epochs) ---
                    decision_dir.mkdir(parents=True, exist_ok=True)
                    build_channel_decision(fid, dec, ch_pct, tool7_bads).to_csv(
                        decision_dir / f'{fid}_channel_decision.tsv', sep='\t', index=False)
                    build_epoch_decision(fid, dec, piv).to_csv(epoch_dec_tsv, sep='\t', index=False)
                    # --- decision TSV (always: feeds the summary, incl. all-rejected / excluded participants) ---
                    decision_summary_row(fid, dec, soi_p, ch_pct, ep_pct,
                                         methods_used=methods_used_str, event_types_used=event_types_used_str,
                                         excluded=bool(excl_reason), exclude_reason=excl_reason,
                                         event_thresholds=thresholds_str,
                                         preproc_fingerprint=settings['preproc_fingerprint']).to_csv(
                        decision_tsv, sep='\t', index=False)
                    # --- the exclusion registry: this tool's automatic exclusions for this participant
                    # (replacing its previous ones, so a re-run with other thresholds can lift them) ---
                    auto_reason = registry_reason(excl_reason, dec)
                    try:
                        PSL.write_registry_rows(reg_folder, '8bis_auto' + run,
                                                [{'file_id': fid, 'reason': auto_reason}] if auto_reason else [],
                                                replace_ids=[fid])
                    except Exception as reg_e:
                        print(f'  ⚠️ {fid}: exclusion registry not updated ({reg_e}).')
                    # --- per-participant report (always, non-fatal) ---
                    try:
                        custom = [s for s in stages_present if s not in AASM_STAGES]
                        fig = plot_decision_heatmap(piv, dec, fid, custom)
                        html = decision_overview_html(fid, dec, soi_p, ch_pct, ep_pct,
                                                      methods_used=methods_used_str,
                                                      event_types_used=event_types_used_str,
                                                      exclude_reason=excl_reason)
                        L.save_report_html(reports_dir / f'{fid}_autoreject_report.html',
                                           f'{fid} - automatic rejection', [('Decision', fig)], html)
                    except Exception as rep_e:
                        print(f'  ⚠️ {fid}: report failed ({rep_e}).')
                    if excl_reason:
                        excluded += 1
                    elif has_output:
                        saved += 1
                    else:
                        all_rejected += 1
                except Exception as e:
                    failed.append((fid, str(e)))
                prog.value += 1
            lbl_prog.value = f'done ({len(parts)}/{len(parts)})'
            UI.finish_progress(prog, len(failed))
            # --- global summary (rebuilt from disk) + failures --- (all in the reports folder)
            summary = _rebuild_global_summary(reports_out)
            if not summary.empty:
                summary = summary.sort_values('file_id')
                reports_out.mkdir(parents=True, exist_ok=True)
                summary.to_csv(reports_out / 'global_autoreject_summary.tsv', sep='\t', index=False)
            if failed:
                reports_out.mkdir(parents=True, exist_ok=True)
                pd.DataFrame(failed, columns=['file_id', 'reason']).to_csv(
                    reports_out / 'autoreject_failed.tsv', sep='\t', index=False)
            print(f'Done. decided={saved}  all-rejected={all_rejected}  '
                  f'excluded (event threshold)={excluded}  skipped={skipped}  failed={len(failed)}')
            if redo:
                print(f'Made again (other settings): {PSL.other_settings_list(differences)}')
            elif differences:
                print(PSL.kept_line(differences, n_processed, n_new))
            if n_not_recorded:
                print(f'ℹ️ {n_not_recorded} participant(s) without their tool 7 flags: their settings could not be compared.')
            print(f'  decisions  -> {decision_out}')
            print(f'  reports    -> {reports_out}')
            if failed:
                print('Failed (could not process):')
                for fid, why in failed:
                    print(f'  ⚠️ {fid}: {why}')
            # --- end-of-run summary table + global report ---
            if not summary.empty:
                # Compact inline table (the full table stays in the global + per-participant reports).
                disp = summary[['file_id', 'n_channels', 'n_channels_rejected',
                                'n_epochs', 'n_epochs_rejected']].copy()
                disp['n_channels_rejected'] = summary.apply(
                    lambda r: (f"{int(r['n_channels_rejected'])} "
                               f"({100 * r['n_channels_rejected'] / r['n_channels']:.1f}%)")
                    if r['n_channels'] else str(int(r['n_channels_rejected'])), axis=1)
                disp['n_epochs_rejected'] = summary.apply(
                    lambda r: f"{int(r['n_epochs_rejected'])} ({r['pct_epochs_rejected']:.1f}%)", axis=1)
                # the share over the reference stages (absent from records written before this column)
                if 'pct_epochs_rejected_ref' in summary.columns:
                    disp['% rejected (reference stages)'] = summary['pct_epochs_rejected_ref'].map(
                        lambda v: '' if pd.isna(v) else f'{v:.1f}%')
                # Show the exclusions of the registry (5bis criteria, manual, tool 7/8, this tool), if any.
                if summary[PSL.COL_EXCLUDED].any():
                    disp['excluded'] = summary[PSL.COL_REASON].fillna('').values
                display(HTML('<h3>Summary - automatic rejection</h3>' + disp.to_html(index=False)))
                try:
                    # a group figure: the excluded participants are left out (and named in the title)
                    kept = summary[~summary[PSL.COL_EXCLUDED]]
                    n_out = len(summary) - len(kept)
                    fig, ax = plt.subplots(figsize=(max(6, min(14, 2 + 0.5 * len(kept))), 3.6),
                                           layout='constrained')
                    # over the reference stages: comparable across participants whatever their amount of
                    # wake (older records without the column fall back to the every-epoch share)
                    pct_ref = (kept['pct_epochs_rejected_ref'].fillna(kept['pct_epochs_rejected'])
                               if 'pct_epochs_rejected_ref' in kept.columns else kept['pct_epochs_rejected'])
                    ax.bar(kept['file_id'].astype(str), pct_ref, color='#34495e')
                    ax.set_ylabel('% reference-stage epochs rejected'); ax.set_xlabel('participant')
                    ax.set_ylim(0, 100)
                    ax.set_title('% reference-stage epochs rejected per participant'
                                 + (f' ({n_out} excluded participant(s) not shown)' if n_out else ''))
                    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
                    buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
                    display(widgets.Image(value=buf.getvalue(), format='png'))
                    L.save_report_html(reports_out / 'autoreject_database_report.html',
                                       'Automatic rejection - database summary',
                                       [('% reference-stage epochs rejected per participant', fig)],
                                       wrap_scroll(summary.to_html(index=False)))
                except Exception as ge:
                    print(f'⚠️ global chart/report failed: {ge}')
        except Exception as e:
            print(f'Run error: {e}')


btn_run.on_click(run_autoreject)
display(widgets.VBox([participant_selector.widget,
                      widgets.HBox([btn_run, participant_selector.run_note], layout=widgets.Layout(align_items='center')),
                      widgets.HBox([prog_row, lbl_prog]), out_run]))


<hr style="height:4px; background-color:black; border:none;">

## References

Channel-first rejection, as in:

- Bigdely-Shamlo N, Mullen T, Kothe C, Su KM, Robbins KA. The PREP pipeline: standardized preprocessing for
  large-scale EEG analysis. Front Neuroinform. 2015;9:16. doi:[10.3389/fninf.2015.00016](https://doi.org/10.3389/fninf.2015.00016)
  (bad channels found before the epochs are judged)
- Nolan H, Whelan R, Reilly RB. FASTER: Fully Automated Statistical Thresholding for EEG artifact Rejection. J Neurosci
  Methods. 2010;192(1):152-62. doi:[10.1016/j.jneumeth.2010.07.015](https://doi.org/10.1016/j.jneumeth.2010.07.015)
  (channel-level, then epoch-level rejection)